### Apache Spark ML basics and examples in a Python environment


In [ ]:
# Install spark

In [26]:
!pip install pyspark
!pip install findspark

import findspark
findspark.init()

from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegressionModel
from pyspark.ml.feature import VectorAssembler

In [ ]:
# Start session

In [2]:
# Creating a spark context class
sc = SparkContext()

# Creating a spark session
spark = SparkSession \
    .builder \
    .appName("Saving and Loading a SparkML Model").getOrCreate()

26/03/01 17:01:42 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 17:01:45 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Download The search term dataset from the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

In [4]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv

--2026-03-01 17:01:49--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/searchterms.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 233457 (228K) [text/csv]
Saving to: ‘searchterms.csv’

searchterms.csv     100%[===================>] 227.99K  --.-KB/s    in 0.04s   

2026-03-01 17:01:49 (5.84 MB/s) - ‘searchterms.csv’ saved [233457/233457]



In [5]:
# Load the csv into a spark dataframe

In [6]:
df = spark.read.csv("searchterms.csv", header=True, inferSchema=True)

In [7]:
# Print the number of rows and columns

In [8]:
rowcount = df.count()
colcount = len(df.columns)
print("Rows:", rowcount)
print("Columns:", colcount)

Rows: 10000
Columns: 4


In [9]:
# Print the top 5 rows

In [10]:
df.show(5, truncate=False)

+---+-----+----+--------------+
|day|month|year|searchterm    |
+---+-----+----+--------------+
|12 |11   |2021|mobile 6 inch |
|12 |11   |2021|mobile latest |
|12 |11   |2021|tablet wifi   |
|12 |11   |2021|laptop 14 inch|
|12 |11   |2021|mobile 5g     |
+---+-----+----+--------------+
only showing top 5 rows



In [11]:
# Find out the datatype of the column searchterm?

In [12]:
datatype = df.schema["searchterm"].dataType
print("Datatype of searchterm:", datatype)

Datatype of searchterm: StringType


In [13]:
# How many times was the term `gaming laptop` searched?

In [14]:
count_gaming_laptop = df.filter(df.searchterm == "gaming laptop").count()
print("Gaming laptop searches:", count_gaming_laptop)

Gaming laptop searches: 499


In [15]:
# Print the top 5 most frequently used search terms?

In [16]:
# Register the DataFrame as a temporary SQL view
df.createOrReplaceTempView("searches")

# Run SQL to get the top 5 most frequent search terms
top5 = spark.sql("""
    SELECT searchterm, COUNT(*) AS cnt
    FROM searches
    GROUP BY searchterm
    ORDER BY cnt DESC
    LIMIT 5
""")

# Show the results
top5.show()

[Stage 8:===================================================>   (186 + 8) / 200]

+-------------+----+
|   searchterm| cnt|
+-------------+----+
|mobile 6 inch|2312|
|    mobile 5g|2301|
|mobile latest|1327|
|       laptop| 935|
|  tablet wifi| 896|
+-------------+----+



In [17]:
# The pretrained sales forecasting model is available at  the below url
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz

In [18]:
# Load the sales forecast model.

In [25]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz 
!tar -xvzf model.tar.gz

model = LinearRegressionModel.load("sales_prediction.model")
print(model)

--2026-03-01 17:11:26--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DB0321EN-SkillsNetwork/Bigdata%20and%20Spark/model.tar.gz
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1490 (1.5K) [application/x-tar]
Saving to: ‘model.tar.gz’

model.tar.gz        100%[===================>]   1.46K  --.-KB/s    in 0s      

2026-03-01 17:11:26 (15.2 MB/s) - ‘model.tar.gz’ saved [1490/1490]

sales_prediction.model/
sales_prediction.model/metadata/
sales_prediction.model/metadata/part-00000
sales_prediction.model/metadata/.part-00000.crc
sales_prediction.model/metadata/_SUCCESS
sales_prediction.model/metadata/._SUCCESS.crc
sales_prediction

In [20]:
# Using the sales forecast model, predict the sales for the year of 2023.

In [31]:
def predict(year):
    assembler = VectorAssembler(inputCols=["year"], outputCol="features")  # Adjusted input column name
    data = [[year]]  # Changed input to reflect height
    columns = ["year"]  # Updated column names for clarity
    df = spark.createDataFrame(data, columns)
    transformed_df = assembler.transform(df).select('features')  # Updated column selection
    predictions = model.transform(transformed_df)
    predictions.show()
    
predict(2023)

+--------+------------------+
|features|        prediction|
+--------+------------------+
|[2023.0]|175.16564294006457|
+--------+------------------+

